# Basic IO positions at expiration

Replication diagnostic, not evidence of strategy validity. The book supplies payoff formulas; contracts and indicative premiums are market examples, not book parameters. The observed index close is context, **not** the official expiration settlement. No fees, discounting or early exercise are included; hypothetical settlement values are nonnegative.

Set `JOSEPH_PILOT_REVISION_ROOT` to the directory containing the three capability revision directories. The default manifest is relative to the repository root; `PILOT_MANIFEST_PATH` can explicitly select another validated manifest.

In [ ]:
import json
import os
from datetime import date
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from joseph_pilot.manifest import Manifest
from joseph_pilot.revision import attach_contracts, load_revision
from joseph_pilot.runtime import Settings

manifest_path = Path(os.environ.get("PILOT_MANIFEST_PATH", "notebooks/book-examples/natenberg-option-volatility-and-pricing/chapter-04-expiration-pnl/manifest.json"))
manifest = Manifest.model_validate_json(manifest_path.read_text())
assert manifest.experiment_id == "replication_diagnostic"
assert version("vectorbt") == manifest.vectorbt_version
np.random.seed(manifest.seed)
root = Path(os.environ["JOSEPH_PILOT_REVISION_ROOT"])
data = {name: load_revision(name, settings=Settings(revision_root=root / name)) for name in ("option_contracts", "option_bars", "index_bars")}
assert set(manifest.inputs) == set(data)
for name, revision in data.items():
    assert revision.pin == manifest.inputs[name], f"revision pin mismatch: {name}"


In [ ]:
symbols = ["IO2611C4000.CFE", "IO2611P4000.CFE"]
observed, expiry = date(2026, 11, 19), date(2026, 11, 20)
contracts = data["option_contracts"].rows
contracts = contracts[contracts.symbol.isin(symbols)].copy()
assert len(contracts) == 2 and set(contracts.symbol) == set(symbols)
assert contracts.list_date.notna().all() and contracts.expiry_date.notna().all()
assert (contracts.list_date <= observed).all() and (contracts.expiry_date == expiry).all()
assert all(str(value)[:7] == "2026-11" for value in contracts.expiry_month)
bars = data["option_bars"].rows
bars = bars.loc[bars.symbol.isin(symbols) & (bars.trade_date == observed), ["symbol", "trade_date", "settle"]]
quotes = attach_contracts(bars, contracts).set_index("symbol")
assert len(quotes) == 2 and set(quotes.index) == set(symbols)
call, put = (quotes.loc[s] for s in symbols)
assert call.option_type.upper() == "C" and put.option_type.upper() == "P"
assert float(call.strike) == float(put.strike) and float(call.multiplier) == float(put.multiplier)
K, multiplier = float(call.strike), float(call.multiplier)
c, p = float(call.settle), float(put.settle)
assert K > 0 and multiplier > 0 and 0 <= c and 0 <= p <= K
index = data["index_bars"].rows
index = index[(index.symbol == "000300.SH") & (index.trade_date == observed)]
assert len(index) == 1
index_close = float(index.iloc[0].close)

def payoffs(s):
    lc = (np.maximum(s - K, 0) - c) * multiplier
    lp = (np.maximum(K - s, 0) - p) * multiplier
    return lc, -lc, lp, -lp

lc, sc, lp, sp = payoffs(np.array([0., K - 100, K, K + 100]))
assert np.allclose(lc, -sc) and np.allclose(lp, -sp)
assert lc[1] == lc[2] == -c * multiplier and lc[3] > lc[2]
assert lp[3] == lp[2] == -p * multiplier and lp[1] > lp[2]
assert lp[0] == (K - p) * multiplier
assert np.isclose(payoffs(K + c)[0], 0) and np.isclose(payoffs(K - p)[2], 0)


In [ ]:
x = np.linspace(0, 2 * K, 401)
fig, ax = plt.subplots(figsize=(9, 5))
for label, values in zip(("Long call", "Short call", "Long put", "Short put"), payoffs(x)):
    ax.plot(x, values, label=label)
ax.axhline(0, color="black", linewidth=0.6)
ax.set(xlabel="Hypothetical expiry settlement index", ylabel="P&L (CNY per contract)")
ax.legend()
plt.show()
summary = {"call_break_even": K + c, "put_break_even": K - p,
           "long_call_max_loss": -c * multiplier, "short_call_max_gain": c * multiplier,
           "long_call_max_gain": "unbounded", "short_call_max_loss": "unbounded",
           "long_put_max_loss": -p * multiplier, "short_put_max_gain": p * multiplier,
           "long_put_max_gain": (K - p) * multiplier, "short_put_max_loss": -(K - p) * multiplier,
           "call_premium": c, "put_premium": p, "strike": K, "multiplier": multiplier,
           "index_close": index_close, "classification": manifest.experiment_id}
print(json.dumps(summary, sort_keys=True))